# Expected Credit Loss under IFRS 9 (Simplified)

**Family:** Credit Risk | **Level:** Intermediate | **Dataset:** `loan_book` | **Data kind:** synthetic | **Module status:** Complete

Stage a loan book, compute ECL, weight macro scenarios and measure the cliff and the sensitivities.

> Data are synthetic. Numbers describe the generator, not any real bank.

## Table of Contents

1. [Problem Statement](#1-problem-statement)
2. [Learning Objectives](#2-learning-objectives)
3. [Fundamental Concepts](#3-fundamental-concepts)
4. [Why This Method?](#4-why-this-method)
5. [Research Foundation](#5-research-foundation)
6. [Mathematical Derivation](#6-mathematical-derivation)
7. [Worked Numerical Example](#7-worked-numerical-example)
8. [Dataset Exploration](#8-dataset-exploration)
9. [Data Preparation](#9-data-preparation)
10. [Implementation from Scratch](#10-implementation-from-scratch)
11. [Implementation using Libraries](#11-implementation-using-libraries)
12. [Model Evaluation](#12-model-evaluation)
13. [Visualization](#13-visualization)
14. [Interpretation](#14-interpretation)
15. [Experiments](#15-experiments)
16. [Results and Comparisons](#16-results-and-comparisons)
17. [Limitations and Model Risk](#17-limitations-and-model-risk)
18. [Practical Banking Application](#18-practical-banking-application)
19. [Exercises](#19-exercises)
20. [References](#20-references)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from risk_lab.reporting.results import find_root, save_config, save_result
from risk_lab.visualization.style import COLORS, apply_style, save_figure

apply_style()
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)
MODULE = "expected_credit_loss"
ROOT = find_root()
SEED = 42
from risk_lab.credit.ecl import (
    assign_stage, balance_profile, cumulative_pd, expected_loss, loan_ecl, marginal_pd,
    scenario_weighted,
)

## 1. Problem Statement

Under IFRS 9 a bank must book a provision **before** a loss happens. The provision is an expected credit loss (ECL),
and its size depends on the stage of the loan:

* **Stage 1.** Credit risk has not risen much since origination. Provision for losses over the next 12 months.
* **Stage 2.** Credit risk has risen significantly. Provision for losses over the whole remaining life.
* **Stage 3.** The loan is credit impaired. Provision for the loss given default on the full exposure.

This notebook builds a simplified ECL engine, applies it to a synthetic loan book of 5000 loans, and measures three things
that decide how a provision behaves: the staging rule, the macro scenario weights, and the jump (cliff) when a loan moves
from stage 1 to stage 2.

**Decision this supports:** how large the provision is, what drives its change, and how sensitive it is to
judgement calls that the standard leaves to the bank.

## 2. Learning Objectives

After this notebook you can:

1. Compute ECL for one loan from PD, LGD, EAD, discounting and a balance profile, by hand and with code.
2. Assign stages with a significant increase in credit risk (SICR) test and the days past due rules.
3. Reconcile the engine with closed form special cases.
4. Aggregate ECL by stage and segment, and compute coverage ratios.
5. Probability weight macro scenarios and measure the effect of weights.
6. Measure how sensitive the provision is to the SICR thresholds and to loan term (the cliff).
7. State which parts of the engine are IFRS 9 principles and which are this laboratory's assumptions.

## 3. Fundamental Concepts

| Term | Meaning |
|---|---|
| PD | Probability of default. Here a 12 month PD per loan |
| LGD | Loss given default, a share of exposure that is lost |
| EAD | Exposure at default, the amount owed when default happens |
| EIR | Effective interest rate, used to discount expected losses |
| SICR | Significant increase in credit risk since origination. Moves a loan to stage 2 |
| DPD | Days past due |
| Coverage ratio | ECL divided by EAD |
| Marginal PD | Probability of defaulting inside year t. Computed here as the difference of two cumulative PDs |

**What is principle and what is assumption.** IFRS 9 requires an unbiased, probability weighted, discounted amount that
uses reasonable and supportable information. It also contains a rebuttable presumption that a loan more than 30 days past due
has a significant increase in credit risk, and a presumption of default at 90 days past due. The relative threshold of 2,
the absolute threshold of 0.01, the constant hazard PD curve, the mid year default timing and the scenario weights in this
notebook are **assumptions of this laboratory**, not requirements.

## 4. Why This Method?

**Why this simplified structure?**

| Option | Strength | Weakness |
|---|---|---|
| Constant hazard from a 12 month PD | One input per loan. Transparent | Ignores that real PD term structures depend on age, macro path and segment |
| Full migration matrix or survival model | Realistic term structure | Needs history that this synthetic book does not have |
| Rule based staging (relative and absolute test plus DPD) | Easy to audit | Creates a cliff at the threshold |

Acceptance criteria, all tested in section 12:

1. For a one year loan in stage 1, ECL equals PD * LGD * EAD discounted half a year (closed form).
2. For stage 3, ECL equals LGD * EAD.
3. For every loan, lifetime ECL is at least 12 month ECL.
4. For every loan, ECL does not exceed EAD.
5. Coverage ratio rises from stage 1 to stage 2 to stage 3 in the total book.
6. Scenario weights sum to 1 and the weighted ECL lies between the lowest and highest scenario ECL.

## 5. Research Foundation


### Regulatory and standard setter sources

- **IFRS 9 Financial Instruments, Section 5.5 Impairment** (IASB, IFRS Foundation, 2014). Status: current. Sets the expected credit loss model: 12 month ECL when credit risk has not increased significantly, lifetime ECL after a significant increase, and lifetime ECL for credit impaired assets. [verified: IFRS Foundation illustrative examples page and IASB staff paper, web search 2026-11-25. The link opens the illustrative examples that accompany the standard. Paragraph numbers were not checked.]
- **Post-implementation Review of IFRS 9 Impairment: summary of feedback and the IASB's responses** (IFRS Foundation Trustees, Due Process Oversight Committee staff paper, 2024). Status: current. Recent official review of how the IFRS 9 impairment model works in practice. Useful for judgement areas such as staging. [verified: IFRS Foundation document, web search 2026-11-25]
- **Basel III: Finalising post-crisis reforms (BCBS d424)** (Basel Committee on Banking Supervision, 2017). Status: integrated into the consolidated Basel Framework. Finalises the Basel III reforms, including the revised standardised approach for credit risk. [verified: BIS page, web search 2026-11-25]
- **SR 26-2: Revised Guidance on Model Risk Management** (Federal Reserve, OCC and FDIC, 2026). Status: current. Current US model risk guidance with a risk based approach. The letter says it is most relevant to organisations above 30 billion dollars in assets and states no effective date. [verified: Federal Reserve page, fetched 2026-11-25]

**How this notebook uses them.** IFRS 9 section 5.5 defines the principles. The IASB post implementation review summarises how entities apply them and where practice differs. Neither document gives numeric thresholds, so every number used here is labelled as an assumption. Paragraph numbers are not quoted because they were not verified.

**Academic findings versus regulatory requirements.** Papers describe what methods can do. Regulations and accounting standards say what a supervised institution must do. This notebook labels which is which, and it does not reproduce any paper's experiment. Data differ, so numbers are not comparable with published results.

## 6. Mathematical Derivation

**Cumulative and marginal PD** with a constant annual hazard from a 12 month PD $p$:

$$F(t) = 1 - (1 - p)^t, \qquad m_t = F(t) - F(t-1) = p(1-p)^{t-1}$$

**Balance profile.** For an annuity with $T$ years and rate $r$ the yearly payment is

$$A = \frac{E\,r}{1 - (1 + r)^{-T}}$$

and the balance at the start of year $t$ follows $B_{t+1} = B_t(1 + r) - A$ with $B_1 = E$.

**ECL** over a horizon of $H$ years, with defaults assumed at the middle of each year:

$$\mathrm{ECL} = \sum_{t=1}^{H} m_t \cdot \mathrm{LGD} \cdot B_t \cdot (1 + r)^{-(t - 0.5)}$$

Stage 1 uses $H = 1$. Stage 2 uses $H = T$. Stage 3 uses $\mathrm{ECL} = \mathrm{LGD} \cdot E$.

**Scenario weighting.** With scenarios $s$ and weights $w_s$ that sum to 1:

$$\mathrm{ECL} = \sum_s w_s \cdot \mathrm{ECL}_s$$

Because ECL is not linear in PD, the ECL at the weighted average PD is generally **not** equal to the weighted average of the ECLs.

## 7. Worked Numerical Example

One loan: EAD 100,000, three years left, EIR 8 percent, annuity repayment, 12 month PD 3 percent, LGD 45 percent.
The table builds each year by hand.

In [2]:
ead, term, eir, pd12, lgd = 100000.0, 3, 0.08, 0.03, 0.45
payment = ead * eir / (1 - (1 + eir) ** (-term))
rows = []
balance = ead
for year in range(1, term + 1):
    cum_now = 1 - (1 - pd12) ** year
    cum_before = 1 - (1 - pd12) ** (year - 1)
    marginal = cum_now - cum_before
    discount = (1 + eir) ** (-(year - 0.5))
    rows.append(
        {"year": year, "balance_start": balance, "marginal_pd": marginal, "discount": discount,
         "loss": marginal * lgd * balance * discount}
    )
    balance = balance * (1 + eir) - payment
by_hand = pd.DataFrame(rows)
print("annual payment:", round(payment, 2))
print(by_hand.round(4).to_string(index=False))
ecl_stage1_hand = by_hand["loss"].iloc[0]
ecl_stage2_hand = by_hand["loss"].sum()
print("stage 1 ECL:", round(ecl_stage1_hand, 2))
print("stage 2 ECL:", round(ecl_stage2_hand, 2))
print("stage 3 ECL:", round(lgd * ead, 2))

annual payment: 38803.35
 year  balance_start  marginal_pd  discount      loss
    1    100000.0000       0.0300    0.9623 1299.0381
    2     69196.6486       0.0291    0.8910  807.3371
    3     35929.0291       0.0282    0.8250  376.4986
stage 1 ECL: 1299.04
stage 2 ECL: 2482.87
stage 3 ECL: 45000.0


In [3]:
assert abs(loan_ecl(pd12, lgd, ead, term, eir, 1) - ecl_stage1_hand) < 1e-8
assert abs(loan_ecl(pd12, lgd, ead, term, eir, 2) - ecl_stage2_hand) < 1e-8
assert abs(loan_ecl(pd12, lgd, ead, term, eir, 3) - lgd * ead) < 1e-8
print("library matches the hand table for all three stages")

library matches the hand table for all three stages


## 8. Dataset Exploration

In [4]:
book = pd.read_csv(ROOT / "datasets" / "synthetic" / "loan_book.csv")
print("loans:", len(book), "total EAD:", f"{book['ead'].sum():,.0f}")
print(book.groupby("segment").agg(loans=("loan_id", "size"), ead=("ead", "sum"), mean_pd=("pd_current", "mean"),
                                  mean_lgd=("lgd", "mean"), mean_term=("remaining_years", "mean")).round(3))
print("share with days past due > 0:", round((book["days_past_due"] > 0).mean(), 4))
print(book["days_past_due"].value_counts().sort_index())

loans: 5000 total EAD: 161,460,130
          loans         ead  mean_pd  mean_lgd  mean_term
segment                                                  
mortgage   1021  32080380.0    0.019     0.202     16.613
retail     2557  83404340.0    0.054     0.549      3.508
sme        1422  45975410.0    0.084     0.450      5.537
share with days past due > 0: 0.054
days_past_due
0      4730
5        51
15       35
35       54
45       39
65       28
95       22
120      30
180      11
Name: count, dtype: int64


## 9. Data Preparation

Stage allocation uses `assign_stage`. A loan is in stage 3 at 90 days past due or more. It is in stage 2 at 30 days past due
or more, or when the current PD is at least twice the origination PD **and** at least 0.01 above it.
Everything else is stage 1. Requiring both the ratio and the gap avoids moving very low PD loans on small absolute changes.

In [5]:
book["stage"] = assign_stage(book["pd_origination"], book["pd_current"], book["days_past_due"])
stage_counts = book["stage"].value_counts().sort_index()
print(stage_counts)
print(pd.crosstab(book["segment"], book["stage"]))
print("EAD share by stage:")
print((book.groupby("stage")["ead"].sum() / book["ead"].sum()).round(4))

stage
1    3869
2    1068
3      63
Name: count, dtype: int64
stage        1    2   3
segment                
mortgage   803  211   7
retail    1978  537  42
sme       1088  320  14
EAD share by stage:
stage
1    0.7769
2    0.2108
3    0.0123
Name: ead, dtype: float64


## 10. Implementation from Scratch

A loop version of the engine, written separately from `loan_ecl`, applied to every loan. It is slow but easy to read.
It is then compared with the library on the whole book.

In [6]:
def my_ecl(pd12, lgd, ead, term, eir, stage):
    if stage == 3:
        return lgd * ead
    horizon = 1 if stage == 1 else term
    payment = ead * eir / (1 - (1 + eir) ** (-term))
    balance = ead
    total = 0.0
    for year in range(1, horizon + 1):
        marginal = (1 - (1 - pd12) ** year) - (1 - (1 - pd12) ** (year - 1))
        total += marginal * lgd * balance * (1 + eir) ** (-(year - 0.5))
        balance = balance * (1 + eir) - payment
    return total


mine = np.array(
    [my_ecl(r.pd_current, r.lgd, r.ead, int(r.remaining_years), r.eir, int(r.stage)) for r in book.itertuples()]
)
lib = np.array(
    [loan_ecl(r.pd_current, r.lgd, r.ead, int(r.remaining_years), r.eir, int(r.stage)) for r in book.itertuples()]
)
print("max gap between loop engine and library:", f"{np.max(np.abs(mine - lib)):.2e}")
assert np.max(np.abs(mine - lib)) < 1e-6
print("total ECL:", f"{lib.sum():,.0f}")

max gap between loop engine and library: 7.28e-12
total ECL: 6,930,960


## 11. Implementation using Libraries

For each loan compute three arrays at once: the 12 month ECL (stage 1 treatment), the lifetime ECL (stage 2 treatment)
and the default ECL (stage 3 treatment). Any staging rule then just picks from these arrays, which makes sensitivity
tests cheap. A PD multiplier scales the PD for macro scenarios.

In [7]:
def all_treatments(frame, pd_multiplier=1.0):
    e1 = np.zeros(len(frame))
    e2 = np.zeros(len(frame))
    e3 = np.zeros(len(frame))
    for i, r in enumerate(frame.itertuples()):
        p = min(r.pd_current * pd_multiplier, 0.999)
        term = int(r.remaining_years)
        e1[i] = loan_ecl(p, r.lgd, r.ead, term, r.eir, 1)
        e2[i] = loan_ecl(p, r.lgd, r.ead, term, r.eir, 2)
        e3[i] = loan_ecl(p, r.lgd, r.ead, term, r.eir, 3)
    return e1, e2, e3


def pick(stages, e1, e2, e3):
    return np.where(stages == 1, e1, np.where(stages == 2, e2, e3))


e1, e2, e3 = all_treatments(book)
book["ecl"] = pick(book["stage"].to_numpy(), e1, e2, e3)
assert np.max(np.abs(book["ecl"].to_numpy() - lib)) < 1e-6
print("treatment arrays reproduce the library ECL")

treatment arrays reproduce the library ECL


## 12. Model Evaluation

Each acceptance criterion from section 4 is tested on the full book. A failed check stops the notebook.

In [8]:
# closed form checks
one_year = loan_ecl(0.04, 0.5, 1000.0, 1, 0.1, 1)
closed = 0.04 * 0.5 * 1000.0 / (1 + 0.1) ** 0.5
c1 = abs(one_year - closed) < 1e-9
c2 = abs(loan_ecl(0.5, 0.4, 2000.0, 5, 0.07, 3) - 0.4 * 2000.0) < 1e-9

# book wide checks
c3 = bool(np.all(e2 >= e1 - 1e-9))
c4 = bool(np.all(book["ecl"] <= book["ead"] + 1e-9))

summary = book.groupby("stage").agg(loans=("loan_id", "size"), ead=("ead", "sum"), ecl=("ecl", "sum"))
summary["coverage_pct"] = 100 * summary["ecl"] / summary["ead"]
c5 = bool(summary["coverage_pct"].is_monotonic_increasing)
print(summary.round(2))
print("total coverage (percent):", round(100 * book["ecl"].sum() / book["ead"].sum(), 3))
checks = {
    "one_year_closed_form": c1,
    "stage3_equals_lgd_times_ead": c2,
    "lifetime_at_least_12m": c3,
    "ecl_not_above_ead": c4,
    "coverage_rises_with_stage": c5,
}
print(checks)
assert all(checks.values())

       loans          ead         ecl  coverage_pct
stage                                              
1       3869  125436780.0  1979712.58          1.58
2       1068   34034210.0  3967345.88         11.66
3         63    1989140.0   983901.88         49.46
total coverage (percent): 4.293
{'one_year_closed_form': True, 'stage3_equals_lgd_times_ead': True, 'lifetime_at_least_12m': True, 'ecl_not_above_ead': True, 'coverage_rises_with_stage': True}


In [9]:
by_segment = book.groupby(["segment", "stage"]).agg(ead=("ead", "sum"), ecl=("ecl", "sum"))
by_segment["coverage_pct"] = 100 * by_segment["ecl"] / by_segment["ead"]
print(by_segment.round(2))

                       ead         ecl  coverage_pct
segment  stage                                      
mortgage 1      24778050.0    59117.59          0.24
         2       7058980.0   409973.35          5.81
         3        243350.0    46759.30         19.21
retail   1      65865260.0  1165870.11          1.77
         2      16252730.0  1857028.02         11.43
         3       1286350.0   727006.22         56.52
sme      1      34793470.0   754724.88          2.17
         2      10722500.0  1700344.51         15.86
         3        459440.0   210136.37         45.74


## 13. Visualization

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
stage_ecl = summary["ecl"] / 1e6
axes[0].bar(stage_ecl.index.astype(str), stage_ecl.values, color=[COLORS["main"], COLORS["third"], COLORS["second"]])
axes[0].set_xlabel("stage")
axes[0].set_ylabel("ECL (millions)")
axes[0].set_title("ECL by stage")

counts = summary["loans"]
axes[1].bar(counts.index.astype(str), counts.values, color=[COLORS["main"], COLORS["third"], COLORS["second"]])
axes[1].set_xlabel("stage")
axes[1].set_ylabel("loans")
axes[1].set_title("Loans by stage")

sorted_ecl = np.sort(book["ecl"].to_numpy())[::-1]
axes[2].plot(np.arange(1, len(sorted_ecl) + 1) / len(sorted_ecl), np.cumsum(sorted_ecl) / sorted_ecl.sum(), color=COLORS["main"])
axes[2].plot([0, 1], [0, 1], color=COLORS["grey"], linestyle=":")
axes[2].set_xlabel("share of loans, largest ECL first")
axes[2].set_ylabel("share of total ECL")
axes[2].set_title("Concentration of ECL")
save_figure(fig, MODULE, "ecl_overview.png")
plt.show()

## 14. Interpretation

How to read the numbers:

* **Coverage ratio by stage.** Stage 1 is small because only 12 months of losses count. Stage 2 is larger because the full life counts and PDs are higher. Stage 3 equals the average LGD of those loans.
* **Concentration.** If a small share of loans holds most of the ECL, the provision depends on a few LGD and PD judgements.
* **Stage 3.** These loans are already in default, so ECL is not an estimate of a probability. It is an estimate of loss severity, and LGD carries all the uncertainty.

In [11]:
top_share = float(sorted_ecl[: int(0.05 * len(sorted_ecl))].sum() / sorted_ecl.sum())
stage3 = book[book["stage"] == 3]
print("share of total ECL held by the top 5 percent of loans:", round(top_share, 4))
print("stage 3 share of total ECL:", round(float(stage3["ecl"].sum() / book["ecl"].sum()), 4))
print("stage 3 coverage equals EAD weighted LGD:", round(float(stage3["ecl"].sum() / stage3["ead"].sum()), 4),
      round(float((stage3["lgd"] * stage3["ead"]).sum() / stage3["ead"].sum()), 4))

share of total ECL held by the top 5 percent of loans: 0.4706
stage 3 share of total ECL: 0.142
stage 3 coverage equals EAD weighted LGD: 0.4946 0.4946


## 15. Experiments

**Experiment A: scenario weights.** Three scenarios multiply every loan's PD: upside 0.8, base 1.0, downside 1.6.
Weights 0.2, 0.5 and 0.3 are an assumption. Staging is held at the base case, which is a simplification. The experiment
also compares the weighted ECL with the ECL at the weighted average PD multiplier.

**Experiment B: SICR thresholds.** Vary the relative threshold (1.5, 2, 3) and the absolute threshold (0.005, 0.01, 0.02).

**Experiment C: the cliff.** Same loan, stage 1 against stage 2, for different remaining terms.

In [12]:
multipliers = {"upside": 0.8, "base": 1.0, "downside": 1.6}
weights = {"upside": 0.2, "base": 0.5, "downside": 0.3}
stages_base = book["stage"].to_numpy()
scenario_totals = {}
for name, mult in multipliers.items():
    s1, s2, s3 = all_treatments(book, mult)
    scenario_totals[name] = float(pick(stages_base, s1, s2, s3).sum())
weighted = scenario_weighted(list(scenario_totals.values()), list(weights.values()))
mean_mult = sum(multipliers[k_] * weights[k_] for k_ in multipliers)
a1, a2, a3 = all_treatments(book, mean_mult)
at_mean_pd = float(pick(stages_base, a1, a2, a3).sum())
scen = pd.DataFrame({"multiplier": multipliers, "weight": weights, "ecl": scenario_totals})
print(scen.round(2))
print("probability weighted ECL:", f"{weighted:,.0f}")
print("ECL at the weighted average PD multiplier:", f"{at_mean_pd:,.0f}")
print("difference (weighted minus at mean PD):", f"{weighted - at_mean_pd:,.0f}")
assert min(scenario_totals.values()) <= weighted <= max(scenario_totals.values())

          multiplier  weight         ecl
upside           0.8     0.2  5902035.02
base             1.0     0.5  6930960.35
downside         1.6     0.3  9719343.84
probability weighted ECL: 7,561,690
ECL at the weighted average PD multiplier: 7,618,042
difference (weighted minus at mean PD): -56,352


In [13]:
rows = []
for rel in [1.5, 2.0, 3.0]:
    for gap in [0.005, 0.01, 0.02]:
        st = assign_stage(book["pd_origination"], book["pd_current"], book["days_past_due"], relative_threshold=rel, absolute_threshold=gap)
        total = float(pick(st, e1, e2, e3).sum())
        rows.append({"relative": rel, "absolute": gap, "stage2_share": float((st == 2).mean()), "total_ecl": total})
sicr = pd.DataFrame(rows)
base_total = float(book["ecl"].sum())
sicr["change_vs_base_pct"] = 100 * (sicr["total_ecl"] / base_total - 1)
print(sicr.round(4).to_string(index=False))

 relative  absolute  stage2_share    total_ecl  change_vs_base_pct
      1.5     0.005        0.3362 7668498.4556             10.6412
      1.5     0.010        0.3136 7615099.6975              9.8708
      1.5     0.020        0.2566 7386416.5989              6.5713
      2.0     0.005        0.2174 6938229.8190              0.1049
      2.0     0.010        0.2136 6930960.3475              0.0000
      2.0     0.020        0.1956 6851946.7469             -1.1400
      3.0     0.005        0.1230 6239739.5708             -9.9729
      3.0     0.010        0.1230 6239739.5708             -9.9729
      3.0     0.020        0.1196 6225588.9508            -10.1771


In [14]:
rows = []
for term_years in [1, 2, 3, 5, 10, 20]:
    s1 = loan_ecl(0.03, 0.45, 100000.0, term_years, 0.08, 1)
    s2 = loan_ecl(0.03, 0.45, 100000.0, term_years, 0.08, 2)
    rows.append({"remaining_years": term_years, "stage1_ecl": s1, "stage2_ecl": s2, "jump_ratio": s2 / s1})
cliff = pd.DataFrame(rows)
print(cliff.round(3).to_string(index=False))

 remaining_years  stage1_ecl  stage2_ecl  jump_ratio
               1    1299.038    1299.038       1.000
               2    1299.038    1904.840       1.466
               3    1299.038    2482.874       1.911
               5    1299.038    3559.638       2.740
              10    1299.038    5833.187       4.490
              20    1299.038    8947.606       6.888


## 16. Results and Comparisons

In [15]:
print(summary.round(2))
print()
print(scen.round(2))
print()
print(checks)
metrics = {
    "total_ead": float(book["ead"].sum()),
    "total_ecl": float(book["ecl"].sum()),
    "coverage_pct": float(100 * book["ecl"].sum() / book["ead"].sum()),
    "by_stage": summary.reset_index().to_dict(orient="records"),
    "scenario_ecl": scenario_totals,
    "scenario_weights": weights,
    "probability_weighted_ecl": weighted,
    "ecl_at_weighted_mean_pd": at_mean_pd,
    "sicr_sensitivity": sicr.to_dict(orient="records"),
    "cliff_by_remaining_years": cliff.to_dict(orient="records"),
    "top_5pct_loans_share_of_ecl": top_share,
    "acceptance_checks": checks,
    "max_gap_loop_vs_library": float(np.max(np.abs(mine - lib))),
}
config = {
    "relative_threshold": 2.0, "absolute_threshold": 0.01, "dpd_stage2": 30, "dpd_stage3": 90,
    "scenario_multipliers": multipliers, "scenario_weights": weights,
    "default_timing": "middle of year", "pd_term_structure": "constant annual hazard",
}
save_config(MODULE, config)
path = save_result(
    MODULE,
    methodology="Simplified IFRS 9 three stage ECL with constant hazard PD, annuity balance profile, mid year default timing and scenario weighting",
    data_kind="synthetic",
    dataset="datasets/synthetic/loan_book.csv",
    n_obs=int(len(book)),
    seed=7,
    parameters=config,
    assumptions=[
        "12 month PD per loan is given and a constant hazard extends it to lifetime",
        "thresholds, scenario multipliers and weights are laboratory assumptions",
        "staging held at the base case in every scenario",
        "balance at the start of each year is the exposure at default",
    ],
    metrics=metrics,
    baseline={"all_loans_treated_as_stage_1_ecl": float(e1.sum()), "note": "12 month ECL with no staging"},
    limitations=[
        "no collateral timing, no cure rates, no prepayment, no off balance sheet exposure",
        "constant hazard ignores the PD term structure",
        "synthetic book, not a real portfolio",
    ],
)
print("saved", path.name)

       loans          ead         ecl  coverage_pct
stage                                              
1       3869  125436780.0  1979712.58          1.58
2       1068   34034210.0  3967345.88         11.66
3         63    1989140.0   983901.88         49.46

          multiplier  weight         ecl
upside           0.8     0.2  5902035.02
base             1.0     0.5  6930960.35
downside         1.6     0.3  9719343.84

{'one_year_closed_form': True, 'stage3_equals_lgd_times_ead': True, 'lifetime_at_least_12m': True, 'ecl_not_above_ead': True, 'coverage_rises_with_stage': True}
saved expected_credit_loss.json


## 17. Limitations and Model Risk

**Limitations**

1. **Constant hazard.** A real lifetime PD depends on loan age, macro path and segment. A constant hazard understates stage 2 ECL when risk rises with time and overstates it when risk falls.
2. **Stage held at base case across scenarios.** In practice SICR is assessed with forward looking information, so staging itself is scenario dependent.
3. **Cliff.** The jump ratio table shows stage 2 ECL can be several times stage 1 ECL for the same loan. The rule based staging transmits a small PD change into a large provision change.
4. **No collateral cash flow timing, cures, prepayments or undrawn commitments.** LGD is a given number.
5. **Parameters are assumptions.** Thresholds, scenario multipliers and weights are lab choices. IFRS 9 leaves the design to the entity, and the IASB post implementation review reports wide practice differences.
6. **Synthetic book.** Segment shares, PDs and LGDs are invented.

**Check against the standard.** Paragraph level references to IFRS 9 are not stated here because they were not verified.
Read section 5.5 of the standard before using any rule in practice.

**Model risk.** The engine is simple and reconciled, but the risk lies in inputs: PD, LGD and the scenario weights drive the result.
Overlay and governance decisions are outside this notebook.

## 18. Practical Banking Application

**What staging adds.** The naive alternative is to give every loan a 12 month provision. The difference between the staged
provision and the naive one is the amount that the lifetime rule adds for stage 2 plus the move to full loss for stage 3.
A finance team reads this as the provision cost of credit deterioration in the book.

**Which loans drive it.** The table lists the ten loans with the largest ECL and how much of the total they hold.

In [16]:
naive = float(e1.sum())
staged = float(book["ecl"].sum())
print("12 month ECL on every loan:", f"{naive:,.0f}")
print("staged ECL:", f"{staged:,.0f}")
print("added by staging:", f"{staged - naive:,.0f}", "which is", round(100 * (staged / naive - 1), 1), "percent more")
top = book.sort_values("ecl", ascending=False).head(10)[["loan_id", "segment", "stage", "ead", "lgd", "pd_current", "ecl"]]
print(top.round(3).to_string(index=False))
print("share of total ECL in these ten loans:", round(float(top["ecl"].sum() / staged), 4))

12 month ECL on every loan: 4,096,495
staged ECL: 6,930,960
added by staging: 2,834,465 which is 69.2 percent more
 loan_id segment  stage      ead   lgd  pd_current        ecl
    4982  retail      3 199500.0 0.658       0.217 131310.900
    3939     sme      3 124990.0 0.478       0.033  59795.216
    1879  retail      3 106720.0 0.513       0.034  54779.376
     817  retail      2 113270.0 0.601       0.362  44372.051
    1455  retail      3  83820.0 0.522       0.350  43745.658
    2253     sme      2 101880.0 0.440       0.817  40814.179
    4009     sme      2 101990.0 0.492       0.583  40727.524
    1735  retail      2 120110.0 0.612       0.295  40108.850
     743     sme      2 166770.0 0.385       0.234  35506.931
    4328  retail      3  66990.0 0.527       0.110  35283.633
share of total ECL in these ten loans: 0.076


## 19. Exercises

1. Change the stage 2 days past due trigger from 30 to 15 days. How many loans move and what is the change in total ECL?
2. Replace the constant hazard with a hazard that rises 10 percent per year. Re derive the marginal PD and recompute the hand table in section 7.
3. Add a fourth scenario with weight 0.1 and multiplier 2.5, and rescale the others so the weights sum to 1. Report the change in weighted ECL.
4. Take the loan in section 7 and find the PD at which stage 2 ECL equals 2 times stage 1 ECL.
5. Suppose LGD is uncertain by plus or minus 10 points. Report the range of total ECL and say which stage it affects most.

## 20. References

Entries are grouped by type. A reference marked NOT VERIFIED must be checked before it is cited.

- **IASB, IFRS Foundation (2014).** IFRS 9 Financial Instruments, Section 5.5 Impairment. Effective for annual periods beginning on or after 1 January 2018. https://www.ifrs.org/content/dam/ifrs/publications/html-standards/english/2026/issued/ifrs9-ie.html
  - Type: regulatory. Status: current. Check: verified: IFRS Foundation illustrative examples page and IASB staff paper, web search 2026-11-25. The link opens the illustrative examples that accompany the standard. Paragraph numbers were not checked.
- **IFRS Foundation Trustees, Due Process Oversight Committee staff paper (2024).** Post-implementation Review of IFRS 9 Impairment: summary of feedback and the IASB's responses. Agenda reference 1C, June 2024. https://cdn.ifrs.org/content/dam/ifrs/meetings/2024/june/dpoc/ap1c-pir-impairment.pdf
  - Type: regulatory. Status: current. Check: verified: IFRS Foundation document, web search 2026-11-25
- **Basel Committee on Banking Supervision (2017).** Basel III: Finalising post-crisis reforms (BCBS d424). Bank for International Settlements, 7 December 2017, consolidated into the Basel Framework. https://www.bis.org/bcbs/publ/d424.htm
  - Type: regulatory. Status: integrated into the consolidated Basel Framework. Check: verified: BIS page, web search 2026-11-25
- **Federal Reserve, OCC and FDIC (2026).** SR 26-2: Revised Guidance on Model Risk Management. 17 April 2026, supersedes SR 11-7 and SR 21-8. https://www.federalreserve.gov/supervisionreg/srletters/sr2602.htm
  - Type: regulatory. Status: current. Check: verified: Federal Reserve page, fetched 2026-11-25